# **Kaggle – DataTops®**
Tu TA ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Aspectos importantes
- Última submission:
    - Mañana: 17 de febrero a las 5pm
    - Tarde: 19 de febrero a las 5pm
- **Enlace de la competición**: https://www.kaggle.com/t/c5cc87b50c4b4770bdc8f5acbe15577d
- **Requisito**: Estar registrado en [Kaggle](https://www.kaggle.com/)

## Métrica:
El error cuadrático medio (RMSE, por sus siglas en inglés) es una medida de la desviación estándar de los residuos (errores de predicción). Los residuos representan la diferencia entre los valores observados y los valores predichos por el modelo. El RMSE indica qué tan dispersos están estos errores: cuanto menor es el RMSE, más cercanas están las predicciones a los valores reales. En otras palabras, el RMSE mide qué tan bien se ajusta la línea de regresión a los datos.


$$ RMSE = \sqrt{\frac{1}{n}\Sigma_{i=1}^{n}{\Big(\frac{d_i -f_i}{\sigma_i}\Big)^2}}$$


## 1. Librerías

In [42]:
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import urllib.request

## 2. Datos

In [ ]:
# Para que funcione necesitas bajarte los archivos de datos de Kaggle
df_portatiles = pd.read_csv("./data/train.csv", index_col="laptop_ID")
df_portatiles.index.name=None

### 2.1 Exploración de los datos

In [ ]:
df_portatiles.info() #Hay mucho string

In [ ]:
df_portatiles.head(5) #Laptop ID --> índice de test csv

In [ ]:
df_portatiles.tail()

In [ ]:
def describe_df(df):
    DATA_TYPE=df.dtypes
    MISSINGS=(df.isna().sum()/len(df)*100).sort_values(ascending=False)
    UNIQUE_VALUES=df.nunique()
    CARDIN=UNIQUE_VALUES/len(df)*100
    describe_df=pd.DataFrame([DATA_TYPE, MISSINGS, UNIQUE_VALUES, CARDIN])
    parametros=["DATA_TYPE", "MISSINGS (%)", "UNIQUE_VALUES", "CARDIN (%)"]
    describe_df.insert(0, "COL_N",parametros)
    return describe_df

In [ ]:
describe_df(df_portatiles).T

In [ ]:
for col in df_portatiles.columns:
    print(df_portatiles[col].value_counts(normalize=True)*100)
    print("-------------------------------------------------------")

In [ ]:
df_portatiles.describe()

In [ ]:
#Hacemos una copia del dataset:
df=df_portatiles.copy()

### 2.3 Definir X e y

In [ ]:
#Nota: considerar cambiar el orden 2.3 <--> 2.4
X = df.drop(['Price_in_euros'], axis=1)
y = df['Price_in_euros'].copy()
X.shape

### 2.4 Dividir X_train, X_test, y_train, y_test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42)

## 3. Procesado de datos

Nuestro target es la columna `Price_in_euros`

In [ ]:
X_train.columns

In [ ]:
#Eliminamos la columna Product ya que tiene demasiada cardinalidad
X_train.drop(columns=["Product"], inplace=True)
X_test.drop(columns=["Product"], inplace=True)

#Columna RAM: Quitamos "GB" y sustituimos por entero
X_train["Ram"]=X_train["Ram"].str.replace('GB', "").astype(int)
X_test["Ram"]=X_test["Ram"].str.replace("GB","")

#Columna weight: quitamos "kg" y lo convertimos a número decimal
X_train["Weight"] = X_train["Weight"].str.replace("kg", "").astype(float)
X_test["Weight"] = X_test["Weight"].str.replace("kg", "").astype(float)



In [ ]:
X_train.isna().sum()

In [ ]:
#Extraemos los números correspondientes al ancho y alto de la resolución:
resolucion_train = X_train["ScreenResolution"].str.extract(r"(\d+)x(\d+)")
resolucion_test = X_test["ScreenResolution"].str.extract(r"(\d+)x(\d+)")

X_train["X_res"] = resolucion_train[0].astype(int)
X_train["Y_res"] = resolucion_train[1].astype(int)

X_test["X_res"] = resolucion_test[0].astype(int)
X_test["Y_res"] = resolucion_test[1].astype(int)

#Creamos la columna de pixeles 
X_train["Pixels"]=X_train["X_res"] * X_train["Y_res"]
X_test["Pixels"]=X_test["X_res"] * X_test["Y_res"]

#Eliminamos X_res e Y_res:
X_train.drop(columns=["X_res", "Y_res"], inplace=True)
X_test.drop(columns=["X_res", "Y_res"], inplace=True)


In [ ]:
X_test.columns

In [ ]:
#Creamos variables binarias para las características extra
X_train['IPS'] = X_train['ScreenResolution'].apply(lambda x: 1 if "IPS" in x else 0)
X_test['IPS'] = X_test['ScreenResolution'].apply(lambda x: 1 if "IPS" in x else 0)
X_train['Touchscreen'] = X_train['ScreenResolution'].apply(lambda x: 1 if 'Touchscreen' in x else 0)
X_test['Touchscreen'] = X_test['ScreenResolution'].apply(lambda x: 1 if 'Touchscreen' in x else 0)

In [ ]:
#Variable CPU: Extraemos el número que está por delante de "GHz":
X_train["Cpu_GHz"] = X_train["Cpu"].str.extract(r'(\d+\.?\d*)GHz').astype(float)
X_test["Cpu_GHz"] = X_test["Cpu"].str.extract(r'(\d+\.?\d*)GHz').astype(float)

In [ ]:
def procesar_cpu(texto):
    if "Intel Core i7" in texto:
        return "Intel Core i7"
    elif "Intel Core i5" in texto:
        return "Intel Core i5"
    elif 'Intel Core i3' in texto:
        return "Intel Core i3"
    elif "Intel" in texto:
        return "Other Intel Processor" # Celeron, Pentium, etc.
    else:
        return "AMD Processor"

X_train["Cpu_Brand"] = X_train["Cpu"].apply(procesar_cpu)
X_test["Cpu_Brand"] = X_test["Cpu"].apply(procesar_cpu)

In [ ]:
X_train.Cpu_Brand.value_counts(normalize=True)

In [ ]:
X_train.columns

In [ ]:
#Variable Memory:
# Aplicamos los cambios a ambos sets
for df_split in [X_train, X_test]:
    # 1. Limpieza básica y estandarización
    df_split["Memory"] = df_split["Memory"].astype(str).replace("\.0", "", regex=True)
    df_split["Memory"] = df_split["Memory"].str.replace("GB", "")
    df_split["Memory"] = df_split["Memory"].str.replace("TB", "000") # Convertimos TB a GB aproximados

    # 2. Creamos columnas binarias para los tipos más comunes
    df_split["Layer1HDD"] = df_split["Memory"].apply(lambda x: 1 if "HDD" in x else 0)
    df_split["Layer1SSD"] = df_split["Memory"].apply(lambda x: 1 if "SSD" in x else 0)
    
    # 3. Extraemos la cantidad numérica
    # Usamos regex para sacar solo los números antes de cualquier texto
    df_split["Memory_Amount"] = df_split["Memory"].str.extract("(\d+)").astype(float)

# Verificamos el resultado en X_train
print(X_train[["Memory", "Memory_Amount", "Layer1SSD", "Layer1HDD"]].head())

In [ ]:
#pd.set_option('display.max_rows', None)
df_portatiles.Gpu.value_counts()

In [ ]:
describe_df(df_portatiles).T

In [ ]:
# Aplicamos la extracción de la marca de la GPU
for df_split in [X_train, X_test]:
    # Extraemos la primera palabra de la columna Gpu, que siempre es la marca
    df_split["Gpu_Brand"] = df_split["Gpu"].apply(lambda x: x.split()[0])
    

# Verificamos que solo tengamos las marcas deseadas
print(X_train["Gpu_Brand"].value_counts())

In [ ]:
#Reducimos la cardinalidad de la variable OpSys:
def categorizar_os(sistema):
    if "Windows" in sistema:
        return "Windows"
    elif "Mac" in sistema or "macOS" in sistema:
        return "Mac"
    elif "Linux" in sistema:
        return "Linux"
    else:
        return "No OS / Others" # Aquí entran "No OS", "Android", "Chrome OS"

# Aplicamos la función a ambos conjuntos
for df_split in [X_train, X_test]:
    df_split["OpSys_Group"] = df_split["OpSys"].apply(categorizar_os)
    

# Verificamos cómo han quedado las nuevas categorías
print(X_train["OpSys_Group"].value_counts())

In [ ]:
pd.set_option('display.max_rows', 10) 

In [ ]:
X_train.columns

In [ ]:
cols_eliminar=["ScreenResolution", "Cpu", "Memory", "Gpu", "OpSys"]
X_train.drop(columns=cols_eliminar, inplace=True)
X_test.drop(columns=cols_eliminar, inplace=True)

In [ ]:
describe_df(X_train).T

In [ ]:
X_train.columns

In [ ]:
describe_df(X_train).T

In [ ]:
y_train

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Unimos temporalmente para calcular la correlación
train_full = X_train.copy()
train_full["Price_in_euros"] = y_train

# 2. Calculamos la correlación de todas las variables con el precio
# Ordenamos de mayor a menor para ver las más importantes arriba
correlations = train_full.corr(numeric_only=True)["Price_in_euros"].sort_values(ascending=False)

# 3. Visualizamos las 20 correlaciones más fuertes (positivas y negativas)
plt.figure(figsize=(10, 8))
correlations.drop("Price_in_euros").head(20).plot(kind="barh")
plt.title("Top 20 Variables con mayor correlación con el Precio")
plt.xlabel("Coeficiente de Correlación de Pearson")
plt.gca().invert_yaxis() # Para que la más alta salga arriba


In [ ]:
y_train_log.hist()

In [ ]:
#Matriz de correlación. Solo numéricas
X_train.corr(numeric_only=True)

In [ ]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

In [ ]:
X_train.columns

In [ ]:
X_train.Pixels.hist(bins=50)

In [ ]:
#Importamos las librerias necesarias:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

In [ ]:
# 1. Lista de columnas actualizada
num_log_features = ["Ram", "Memory_Amount", "Pixels"] 
num_features = ["Inches", "Weight", "Cpu_GHz", "IPS", "Touchscreen", "Layer1HDD", "Layer1SSD"]
cat_features = ["Company", "TypeName", "Cpu_Brand", "Gpu_Brand", "OpSys_Group"]

# 2. Pipelines simplificados (sin SimpleImputer)
cat_pipeline = Pipeline([
    ("OHEncoder", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False))
])

num_pipeline_log = Pipeline([
    ("Apply_Logarithm", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("SScaler", StandardScaler())
])

num_pipeline = Pipeline([
    ("SScaler", StandardScaler())
])

# 3. ColumnTransformer
preprocessing = ColumnTransformer([
    ("Process_Numeric_Log", num_pipeline_log, num_log_features),
    ("Process_Numeric", num_pipeline, num_features),
    ("Process_Categorical", cat_pipeline, cat_features)
], remainder="passthrough")

-----------------------------------------------------------------------------------------------------------------

## 4. Modelado

### 4.1 Baseline de modelos


In [ ]:
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, GridSearchCV

# Pipelines individuales
ridge_pipe = Pipeline([
    ("Preprocesado", preprocessing),
    ("Modelo", Ridge())
])

rf_pipe = Pipeline([
    ("Preprocesado", preprocessing),
    ("Modelo", RandomForestRegressor(random_state=42))
])

xgb_pipe = Pipeline([
    ("Preprocesado", preprocessing),
    ("Modelo", XGBRegressor(random_state=42))
])




### 4.2 Sacar métricas, valorar los modelos

Recuerda que en la competición se va a evaluar con la métrica de ``RMSE``.

In [ ]:
# Evaluación rápida con Cross Validation (usando R²)
for name, pipe in zip(["Ridge", "RandomForest", "Xgboost"], 
                     [ridge_pipe, rf_pipe, xgb_pipe]):
    resultado = cross_val_score(pipe, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error")
    print(f"{name}: {-np.mean(resultado):.4f} {resultado}")


### 4.3 Optimización (up to you 🫰🏻)

In [ ]:
#Definición de hiperparámetros:
param_ridge = {
    "Modelo__alpha": [0.1, 1.0, 10.0]
}

param_rf = {
    "Modelo__n_estimators": [100, 200],
    "Modelo__max_depth": [None, 10, 20],
    "Modelo__max_features": ["sqrt", None]
}

param_xgb = {
    "Modelo__n_estimators": [100, 200, 500],
    "Modelo__learning_rate": [0.01, 0.05, 0.1],
    "Modelo__max_depth": [3, 5, 7],
    "Modelo__subsample": [0.8, 1.0]
}

In [ ]:
#Inicialización de los GridSearch
cv = 5
scoring_metric = "neg_root_mean_squared_error"

gs_ridge = GridSearchCV(ridge_pipe, param_ridge, cv=cv, scoring=scoring_metric, n_jobs=-1)
gs_rf = GridSearchCV(rf_pipe, param_rf, cv=cv, scoring=scoring_metric, n_jobs=-1)
gs_xgb = GridSearchCV(xgb_pipe, param_xgb, cv=cv, scoring=scoring_metric, n_jobs=-1, verbose=1)

# 4. Ejecución y resultados
pipe_grids = {
    "Ridge": gs_ridge,
    "RandomForest": gs_rf,
    "XGBoost": gs_xgb
}

In [ ]:
best_models ={}

for name, gs in pipe_grids.items():
    print(f"Entrenando GridSearch para {name}...")
    gs.fit(X_train, y_train)
    best_models[name] = gs.best_estimator_
    print(f"Mejor RMSE en CV para {name}: {-gs.best_score_:.4f}")
    print(f"Mejores parámetros: {gs.best_params_}\n")

-----------------------------------------------------------------

## Una vez listo el modelo, toca predecir ``test.csv``

**RECUERDA: APLICAR LAS TRANSFORMACIONES QUE HAYAS REALIZADO EN `train.csv` a `test.csv`.**


Véase:
- Estandarización/Normalización
- Eliminación de Outliers
- Eliminación de columnas
- Creación de columnas nuevas
- Gestión de valores nulos
- Y un largo etcétera de técnicas que como Data Scientist hayas considerado las mejores para tu dataset.

## 1. Carga los datos de `test.csv` para predecir.


In [ ]:
#RECORDAR HACER LAS MISMAS TRANSFORMACIONES

In [ ]:
X_pred = pd.read_csv("./data/test.csv",index_col="laptop_ID")
df.index.name=None #establecer mejor laptop_id como indice
X_pred.head()

In [ ]:
X_pred.tail()

In [ ]:
X_pred.info()

 ## 2. Replicar el procesado para ``test.csv``

In [ ]:
X_pred

In [ ]:
X_pred["Ram"]=X_pred["Ram"].str.replace("GB","")
X_pred["Ram"]=X_pred["Ram"].astype(int)

In [ ]:
#a=X_pred["laptop_id"]

In [ ]:
model=rf

predictions_submit = model.predict(X_pred[features])
predictions_submit

**¡OJO! ¿Por qué me da error?**

IMPORTANTE:

- SI EL ARRAY CON EL QUE HICISTEIS `.fit()` ERA DE 4 COLUMNAS, PARA `.predict()` DEBEN SER LAS MISMAS
- SI AL ARRAY CON EL QUE HICISTEIS `.fit()` LO NORMALIZASTEIS, PARA `.predict()` DEBÉIS NORMALIZARLO
- TODO IGUAL SALVO **BORRAR FILAS**, EL NÚMERO DE ROWS SE DEBE MANTENER EN ESTE SET, PUES LA PREDICCIÓN DEBE TENER **391 FILAS**, SI O SI

**Entonces, si al cargar los datos de ``train.csv`` usaste `index_col=0`, ¿tendré que hacer lo también para el `test.csv`?**

In [ ]:
# ¿Qué opináis? Si hay que hacerlo
# ¿Sí, no?

![wow.jpeg](attachment:wow.jpeg)

## 3. **¿Qué es lo que subirás a Kaggle?**

**Para subir a Kaggle la predicción esta tendrá que tener una forma específica.**

En este caso, la **MISMA** forma que `sample_submission.csv`.

In [ ]:
sample = pd.read_csv("data/sample_submission.csv") #ïndice laptop ID

In [ ]:
sample #Formato que hay que subir

In [ ]:
sample.head()

In [ ]:
sample.shape

## 4. Mete tus predicciones en un dataframe llamado ``submission``.

In [ ]:
#¿Cómo creamos la submission?
submission = pd.DataFrame({"laptop_ID": X_pred.index , "Price_in_euros" : predictions_submit})

In [ ]:
submission.head()

In [ ]:
submission.shape

## 5. Pásale el CHEQUEADOR para comprobar que efectivamente está listo para subir a Kaggle.

In [ ]:
def chequeador(df_to_submit):
    """
    Esta función se asegura de que tu submission tenga la forma requerida por Kaggle.

    Si es así, se guardará el dataframe en un `csv` y estará listo para subir a Kaggle.

    Si no, LEE EL MENSAJE Y HAZLE CASO.

    Si aún no:
    - apaga tu ordenador,
    - date una vuelta,
    - enciendelo otra vez,
    - abre este notebook y
    - leelo todo de nuevo.
    Todos nos merecemos una segunda oportunidad. También tú.
    """
    if df_to_submit.shape == sample.shape:
        if df_to_submit.columns.all() == sample.columns.all():
            if df_to_submit.laptop_ID.all() == sample.laptop_ID.all():
                print("You're ready to submit!")
                df_to_submit.to_csv("submission.csv", index = False) #muy importante el index = False
                urllib.request.urlretrieve("https://www.mihaileric.com/static/evaluation-meme-e0a350f278a36346e6d46b139b1d0da0-ed51e.jpg", "gfg.png")
                img = Image.open("gfg.png")
                img.show()
            else:
                print("Check the ids and try again")
        else:
            print("Check the names of the columns and try again")
    else:
        print("Check the number of rows and/or columns and try again")
        print("\nMensaje secreto del TA: No me puedo creer que después de todo este notebook hayas hecho algún cambio en las filas de `test.csv`. Lloro.")

In [ ]:
chequeador(submission)